In [ ]:
# 生成文件夹内视频信息表。万计的视频文件。
import os
import csv
import subprocess
from pathlib import Path

def get_video_info(video_path: Path):
    """使用 ffprobe 获取视频分辨率、帧率、时长、比特率"""
    cmd = [
        'ffprobe',
        '-v', 'error',
        '-select_streams', 'v:0',
        '-show_entries',
        'stream=width,height,r_frame_rate,bit_rate',
        '-show_entries',
        'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(video_path)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        lines = result.stdout.strip().split('\n')
        width = lines[0]
        height = lines[1]
        framerate = eval(lines[2]) if '/' in lines[2] else float(lines[2])
        bitrate = int(lines[3]) if lines[3].isdigit() else 0
        duration = float(lines[4])
        resolution = f"{width}x{height}"
        return resolution, duration, round(framerate, 2), round(bitrate / 1000), 0  # 使用次数初始为0
    except Exception as e:
        raise RuntimeError(f"[错误] 获取视频信息失败：{video_path.name} -> {e}")

def safe_str(s):
    """防止写入CSV时报错，替换无法编码的字符"""
    return s.encode('utf-8', errors='replace').decode('utf-8')

def scan_video_folders(root_dir: Path, output_csv: Path, error_csv: Path):
    """扫描所有视频文件，写入正常和异常表"""
    with open(output_csv, 'w', newline='', encoding='utf-8') as main_file, \
         open(error_csv, 'w', newline='', encoding='utf-8', errors='replace') as error_file:

        main_writer = csv.writer(main_file)
        error_writer = csv.writer(error_file)

        headers = ["子文件夹", "文件名", "分辨率", "时长（秒）", "帧率", "比特率（kbps）", "使用次数"]
        main_writer.writerow(headers)
        error_writer.writerow(headers)

        for subdir, _, files in os.walk(root_dir):
            subfolder = Path(subdir).relative_to(root_dir)
            for file in files:
                if file.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.flv')):
                    video_path = Path(subdir) / file
                    try:
                        resolution, duration, fps, bitrate, usage = get_video_info(video_path)
                        main_writer.writerow([str(subfolder), file, resolution, duration, fps, bitrate, usage])
                        print(f"[✓] {subfolder}/{file} -> {resolution}, {duration:.1f}s, {fps}fps, {bitrate}kbps")
                    except Exception as e:
                        # 写入异常表，字符不合法的替换为 �
                        error_writer.writerow([
                            safe_str(str(subfolder)),
                            safe_str(file),
                            "unknown", 0, 0.0, 0, 0
                        ])
                        print(f"[×] {subfolder}/{file} -> 异常，写入错误表")

    print(f"\n✅ 正常视频信息已保存至：{output_csv}")
    print(f"⚠️ 异常视频信息已保存至：{error_csv}")

# 示例调用
if __name__ == "__main__":
    input_dir = Path("")  # <-- 替换成你的文件夹
    output_csv = input_dir / "video_info.csv"
    error_csv = input_dir / "video_errors.csv"
    scan_video_folders(input_dir, output_csv, error_csv)


In [ ]:
# 文件数统计，基于这个 CSV 文件作为后续任务的索引或任务规划文件
import os
import csv
from pathlib import Path
from prettytable import PrettyTable  # pip install prettytable（可选）

# 设置根目录
root_dir = Path("")  # <-- 替换成你的文件夹

# 构建表格（可选美化）
table = PrettyTable()
table.field_names = ["子文件夹名", "文件数"]

# 准备 CSV 数据
csv_rows = [("子文件夹名", "文件数")]

# 遍历每个子文件夹
for subfolder in sorted(root_dir.iterdir()):
    if subfolder.is_dir():
        file_count = sum(1 for _ in subfolder.iterdir() if _.is_file())
        table.add_row([subfolder.name, file_count])
        csv_rows.append((subfolder.name, file_count))

# 打印表格到控制台
print(table)

# 保存为 CSV 文件
csv_path = root_dir / "文件数量统计.csv"
with open(csv_path, mode="w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerows(csv_rows)

print(f"\n✅ 已保存 CSV 文件到: {csv_path}")


In [ ]:
import pandas as pd
from moviepy.video.io.VideoFileClip import VideoFileClip
from pathlib import Path
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# 配置路径
# root_dir = Path("G:/")
csv_path = root_dir / "文件数量统计.csv"

# 线程锁，保护 DataFrame 写操作
lock = threading.Lock()

# 读取CSV为DataFrame
df = pd.read_csv(csv_path)

# 如果没有这两列则创建
if "视频尺寸列表" not in df.columns:
    df["视频尺寸列表"] = ""
if "尺寸总时长(秒)" not in df.columns:
    df["尺寸总时长(秒)"] = ""

# 获取需要处理的索引列表
pending_indices = [
    idx for idx, row in df.iterrows()
    if (pd.isna(row["视频尺寸列表"]) or str(row["视频尺寸列表"]).strip() == "") and int(row["文件数"]) < 1000
]

print(f"📦 需要处理的文件夹数量：{len(pending_indices)}")

def process_folder(idx):
    row = df.loc[idx]
    folder_name = row["子文件夹名"]
    folder_path = root_dir / folder_name

    if not folder_path.exists():
        print(f"❌ 文件夹不存在: {folder_path}")
        return

    resolution_set = set()
    duration_dict = {}

    for file in folder_path.glob("*"):
        if file.suffix.lower() not in [".mp4", ".mov", ".avi", ".mkv"]:
            continue
        try:
            clip = VideoFileClip(str(file))
            w, h = clip.size
            dur = clip.duration
            res_key = f"{w}x{h}"
            resolution_set.add(res_key)
            duration_dict[res_key] = duration_dict.get(res_key, 0) + dur
            clip.reader.close()
            clip.close()
        except Exception as e:
            print(f"⚠️ 处理失败: {file.name} - {e}")

    # 线程安全写入 DataFrame 和 CSV
    with lock:
        df.at[idx, "视频尺寸列表"] = sorted(list(resolution_set))
        df.at[idx, "尺寸总时长(秒)"] = str({k: round(v, 2) for k, v in duration_dict.items()})
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"✅ 已写入: {folder_name}")

# 多线程处理
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(process_folder, idx) for idx in pending_indices]
    for future in as_completed(futures):
        pass  # 可以加进度条或者错误处理日志

print(f"\n✅ 所有任务完成，已更新 CSV：{csv_path}")


In [ ]:
from moviepy.video.compositing.CompositeVideoClip import clips_array
from moviepy.video.io.VideoFileClip import VideoFileClip

target_fps = 30

# 加载三个竖屏视频
clip1 = VideoFileClip("a.mp4").resize((720, 1280)).set_fps(target_fps)
clip2 = VideoFileClip("b.mp4").resize((720, 1280)).set_fps(target_fps)
clip3 = VideoFileClip("c.mp4").resize((720, 1280)).set_fps(target_fps)

# 横向拼接成一个 2160x1280 的视频
final_clip = clips_array([[clip1, clip2, clip3]])

# 保存成文件
final_clip.write_videofile("horizontal_output.mp4", fps=target_fps,codec="libx264", audio=False)


1.将文件夹内不同分辨率的视频按顺序分别合并成一个长视频，不保留音频，注意帧率一致的问题
2.将三个竖屏视频合并成一个横屏视频。横屏分为左中右三个视频轨道来顺序放置竖屏视频。

In [ ]:
# A/
# ├── B1/
# │   ├── v1.mp4
# │   ├── v2.mp4
# ├── B2/
# │   ├── x1.mp4
# │   ├── x2.mp4

# merged/
# ├── B1_merged.mp4
# ├── B2_merged.mp4

import os
from moviepy.video.io.VideoFileClip import VideoFileClip
from moviepy.video.compositing.CompositeVideoClip import concatenate_videoclips

# 参数配置
# input_root = "G:\\"
# output_root = "G:\\"
target_size = (720, 1280)  # 宽x高
target_fps = 30

os.makedirs(output_root, exist_ok=True)

for subfolder in sorted(os.listdir(input_root)):
    print(f"Processing: {subfolder}")
    subfolder_path = os.path.join(input_root, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    clips = []
    for file in sorted(os.listdir(subfolder_path)):
        if not file.lower().endswith((".mp4", ".mov", ".avi", ".mkv")):
            continue

        video_path = os.path.join(subfolder_path, file)
        print(f"Processing: {video_path}")

        try:
            clip = VideoFileClip(video_path)
            clip = clip.resize(target_size).set_fps(target_fps).without_audio()
            clips.append(clip)
        except Exception as e:
            print(f"⚠️ Skipped {video_path}: {e}")

    if clips:
        final = concatenate_videoclips(clips, method="compose")
        output_path = os.path.join(output_root, f"{subfolder}_merged.mp4")
        final.write_videofile(output_path, fps=target_fps, codec="libx264", audio=False)
        print(f"✅ Saved: {output_path}")
    else:
        print(f"⚠️ No valid videos found in {subfolder_path}")


In [ ]:
import os
from pathlib import Path
from moviepy.video.io.VideoFileClip import VideoFileClip
from moviepy.video.compositing.CompositeVideoClip import concatenate_videoclips

# 参数配置
# input_root = Path(r"G:\")
# output_root = Path(r"G:\")
target_size = (720, 1280)
target_fps = 30

output_root.mkdir(parents=True, exist_ok=True)

for subfolder in sorted(input_root.iterdir()):
    if not subfolder.is_dir():
        continue

    subfolder = input_root
    print(f"📁 Processing folder: {subfolder.name}")
    clips = []

    for file in sorted(subfolder.iterdir()):
        if file.suffix.lower() not in [".mp4", ".mov", ".avi", ".mkv"]:
            continue

        print(f"🎞️ Processing video: {file.name}")
        try:
            clip = VideoFileClip(str(file))
            clips.append(clip)
        except Exception as e:
            print(f"⚠️ Skipped {file.name}: {e}")

    if clips:
        try:
            final = concatenate_videoclips(clips, method="compose")
            output_path = output_root / f"{subfolder.name}_merged.mp4"
            print(f"💾 Saving to: {output_path}")
            final.write_videofile(str(output_path), fps=target_fps, codec="libx264", audio=False)
            print(f"✅ Done: {output_path}")
            final.close()
            for clip in clips:
                clip.close()
        except Exception as e:
            print(f"❌ Failed to save {subfolder.name}_merged.mp4: {e}")
    else:
        print(f"⚠️ No valid videos in {subfolder}")
